In [0]:
# MAGIC %md
# MAGIC # 01 — Ingestion API ERP → Volume (version simple)
# MAGIC Lit l'API ERP AgriMeca (Azure Container Apps) et dépose chaque page JSON dans
# MAGIC `/Volumes/supplychain/landing/erp/<entité>/ingest_date=AAAAMMJJ/run_id=<id>/page_000001.json`.
# MAGIC Les secrets viennent de Key Vault via le scope `erp` : rien n'est écrit en clair.
 
# COMMAND ----------

In [0]:
# cellule 1 configuration
import json
import time
import uuid
from datetime import datetime, timezone
import requests

BASE_URL = dbutils.secrets.get ("erp", "erp-base-url") # récupération de secret scope
CLIENT_ID = "agrimeca-data"
CLIENT_SECRET = dbutils.secrets.get ("erp", "erp-client-secret") # lire le mot de passe dans Keyvault en passant par le secret scope
LANDING = "/Volumes/supplychain/landing/erp"

print(BASE_URL) # affichage de la base url, databricks masque le secret



In [0]:
# cellule 2 - jeton OAuth (valable 15 minutes)
def get_token() -> str:
    r = requests.post(
        f"{BASE_URL}/auth/token",
        json={"client_id": CLIENT_ID, "client_secret": CLIENT_SECRET},
        timeout=30,
    )
    r.raise_for_status()  # lève une exception si HTTP a une erreur (code entre 400 et 599)
    return r.json()["access_token"]


headers = {"Authorization": f"Bearer {get_token()}"}
print("Jeton obtenu")

### VERIFICATION DE LA DISPONIBILITE DE L'API 

In [0]:
print("finit par / :", BASE_URL.endswith("/"))
print("longueur    :", len(BASE_URL))
print(" ".join(BASE_URL))   # affiche l'URL lettre par lettre, ce qui contourne le masquage

In [0]:
r = requests.get(f"{BASE_URL}/docs", timeout=60)
print(r.status_code)
print(r.text[:300])

In [0]:
# cellule 3 - premier appel pour observer la réponse
r= requests.get(f"{BASE_URL}/v1/suppliers",headers=headers, params={"limit":5}, timeout=30)
r.raise_for_status()
page = r.json()

print("meta : ", page["meta"])
print("paging : ", page["paging"])
print("1er  fournisseur :", json.dumps(page["data"][0], indent=2, ensure_ascii=False))

In [0]:
# cellule 4 - pagination et ecriture de chaque page dans le Volume
def get_page(entity: str) -> None:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
    folder = f"{LANDING}/{entity}/ingest_date={run_id[:8]}/run_id={run_id}"  # 8 premiers caractères de run_id = la date
    dbutils.fs.mkdirs(folder)

    cursor = None  # pas de marque-page au départ (Tour 1 : GET /v1/suppliers?limit=1000)
    n_page = 0     # compteur de pages
    n_rec = 0      # compteur d'enregistrements

    while True:  # on boucle tant qu'il y a encore des pages
        params = {"limit": 1000}  # à chaque appel, on récupère 1000 enregistrements
        if cursor:  # je n'envoie le marque-page que si j'en ai un
            params["cursor"] = cursor  # Tour 2 : GET /v1/suppliers?limit=1000&cursor=...
        r = requests.get(f"{BASE_URL}/v1/{entity}", headers=headers, params=params, timeout=60)
        if r.status_code == 401:                                  # jeton expiré ou inconnu
            headers["Authorization"] = f"Bearer {get_token()}"    # on en redemande un
            continue                                              # et on refait le même appel

        if r.status_code == 429:  # quota dépassé : attendre le délai demandé
            time.sleep(int(r.headers.get("Retry-After", 5)))  # 5 secondes par défaut
            continue
        if r.status_code in (500, 503):  # erreur passagère simulée par l'ERP
            time.sleep(2)
            continue
        r.raise_for_status()

        page = r.json()
        n_page += 1                     # on incrémente le compteur de pages
        n_rec += page["meta"]["count"]  # on cumule le nombre d'enregistrements
        with open(f"{folder}/page_{n_page:06d}.json", "w") as f:  # page_000001.json, ouvert en écriture (w)
            json.dump(
                {"_run_id": run_id, "_fetched_at": datetime.now(timezone.utc).isoformat(), **page},
                f,
            )
        if not page["paging"]["has_more"]:  # pas de page suivante : on sort de la boucle
            break
        cursor = page["paging"]["next_cursor"]  # marque-page pour la prochaine page

    print(f"{entity} : {n_page} pages, {n_rec} enregistrements -> {folder}")


get_page("suppliers")

In [0]:
for day in dbutils.fs.ls(f"{LANDING}/suppliers"):
    for run in dbutils.fs.ls(day.path):
        files = dbutils.fs.ls(run.path)
        print(day.name, run.name, "→", len(files), "fichier(s)")

### Suppression des runs vides


In [0]:
# Ménage : suppression des runs vides (essais ratés)
for day in dbutils.fs.ls(f"{LANDING}/suppliers"):
    for run in dbutils.fs.ls(day.path):
        if len(dbutils.fs.ls(run.path)) == 0:
            dbutils.fs.rm(run.path, recurse=True)
            print("supprimé :", run.name)